# 08 - Agentic Workflow for Financial Tweet Classification
## Extra Challenge 2 (+1.50 pts)

A conversational agent that orchestrates **three** classifiers with **non-trivial, adaptive routing**
(not simple voting). Runs fully offline via a deterministic `RuleBasedAgent` (no API key needed);
an optional LangChain ReAct back-end is used when an LLM key is available.

### Tools
1. **VADER** - fast lexical baseline
2. **LightGBM + SBERT** - ML model (retrained on SBERT embeddings)
3. **FinBERT** - financial-domain transformer (Araci, 2019)

### Orchestration
```
Tweet -> VADER (quick baseline)
          |- strong signal (|compound| > 0.3) -> trust VADER
          |- weak signal -> LightGBM + SBERT
                             |- agrees with VADER -> final verdict
                             |- disagrees        -> FinBERT (domain tiebreaker)
```


In [1]:
import os, sys
# Run from the project root (group_xx) regardless of where the kernel starts
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

from src.agent import (
    classify_with_vader, classify_with_lgbm, classify_with_finbert,
    build_langchain_agent, RuleBasedAgent, MockAgent,
)
print('Agent module loaded. CWD:', os.getcwd())

Agent module loaded. CWD: C:\Users\tiago\OneDrive - NOVAIMS\Ambiente de Trabalho\textmining\group_xx


In [2]:
# The three tools individually
test_tweets = [
    '$AAPL beats Q4 earnings by 15%, revenue up 8% YoY, raises guidance',
    '$TSLA misses estimates, shares down 12%, multiple downgrades',
    'Market trading volume relatively stable today, no major catalysts',
    '$NVDA mixed results: GPU sales strong but data center growth disappoints',
]
for tweet in test_tweets:
    print('Tweet:', tweet[:70])
    print('  VADER:   ', classify_with_vader(tweet))
    print('  LightGBM:', classify_with_lgbm(tweet))
    print('  FinBERT: ', classify_with_finbert(tweet))
    print()

Tweet: $AAPL beats Q4 earnings by 15%, revenue up 8% YoY, raises guidance
  VADER:    Neutral (2) — VADER compound=0.000 [pos=0.00, neg=0.00, neu=1.00]


C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LightGBM: Bullish (1) — confidence=0.998 [Bearish=0.000, Bullish=0.998, Neutral=0.002]


  FinBERT:  Bullish (1) — FinBERT=positive confidence=0.952

Tweet: $TSLA misses estimates, shares down 12%, multiple downgrades
  VADER:    Bullish (1) — VADER compound=0.077 [pos=0.22, neg=0.19, neu=0.59]
  LightGBM: Bearish (0) — confidence=1.000 [Bearish=1.000, Bullish=0.000, Neutral=0.000]
  FinBERT:  Bearish (0) — FinBERT=negative confidence=0.971

Tweet: Market trading volume relatively stable today, no major catalysts
  VADER:    Neutral (2) — VADER compound=0.000 [pos=0.19, neg=0.19, neu=0.61]
  LightGBM: Neutral (2) — confidence=0.537 [Bearish=0.003, Bullish=0.459, Neutral=0.537]
  FinBERT:  Bullish (1) — FinBERT=positive confidence=0.801

Tweet: $NVDA mixed results: GPU sales strong but data center growth disappoin
  VADER:    Bullish (1) — VADER compound=0.285 [pos=0.33, neg=0.20, neu=0.47]
  LightGBM: Bullish (1) — confidence=0.602 [Bearish=0.033, Bullish=0.602, Neutral=0.365]
  FinBERT:  Bullish (1) — FinBERT=positive confidence=0.790



C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [3]:
# Build the agent (deterministic RuleBasedAgent without an LLM key)
api_key = os.environ.get('OPENAI_API_KEY', None)
agent = build_langchain_agent(openai_api_key=api_key)
print('Agent type:', type(agent).__name__)

LangChain agent unavailable (No module named 'langchain').
Using RuleBasedAgent (deterministic offline orchestration; no LLM key).
Agent type: RuleBasedAgent


In [4]:
# Adaptive-routing demonstration with explanations
print('=' * 64)
print('FINANCIAL TWEET SENTIMENT AGENT')
print('=' * 64)
for tweet in test_tweets:
    print('\nINPUT:', tweet)
    print('-' * 44)
    print(agent.invoke({'input': tweet})['output'])

FINANCIAL TWEET SENTIMENT AGENT

INPUT: $AAPL beats Q4 earnings by 15%, revenue up 8% YoY, raises guidance
--------------------------------------------


C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Reasoning trace:
  1. VADER baseline -> Neutral (compound=+0.000)
  2. weak signal -> LightGBM+SBERT -> Bullish (conf=1.00)
  3. disagreement -> FinBERT tiebreaker -> Bullish (conf=0.95)
FINAL VERDICT: Bullish (class=1)
Why: VADER and LightGBM disagreed; FinBERT (financial-domain expert) broke the tie

INPUT: $TSLA misses estimates, shares down 12%, multiple downgrades
--------------------------------------------
Reasoning trace:
  1. VADER baseline -> Bullish (compound=+0.077)
  2. weak signal -> LightGBM+SBERT -> Bearish (conf=1.00)
  3. disagreement -> FinBERT tiebreaker -> Bearish (conf=0.97)
FINAL VERDICT: Bearish (class=0)
Why: VADER and LightGBM disagreed; FinBERT (financial-domain expert) broke the tie

INPUT: Market trading volume relatively stable today, no major catalysts
--------------------------------------------
Reasoning trace:
  1. VADER baseline -> Neutral (compound=+0.000)
  2. weak signal -> LightGBM+SBERT -> Neutral (conf=0.54)
FINAL VERDICT: Neutral (class=2)
Why:

C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
# Conversational follow-up - the agent recalls its last analysis from memory
demo = '$AMZN crushes earnings, AWS reaccelerates, shares surge after hours'
print('INPUT:', demo)
print(agent.invoke({'input': demo})['output'])
if hasattr(agent, 'chat'):
    print('\n--- follow-up question (conversational memory) ---')
    print('USER: why did you classify that as Bullish?')
    print(agent.chat('why did you classify that as Bullish?'))

INPUT: $AMZN crushes earnings, AWS reaccelerates, shares surge after hours


Reasoning trace:
  1. VADER baseline -> Bearish (compound=-0.178)
  2. weak signal -> LightGBM+SBERT -> Bullish (conf=0.92)
  3. disagreement -> FinBERT tiebreaker -> Bullish (conf=0.78)
FINAL VERDICT: Bullish (class=1)
Why: VADER and LightGBM disagreed; FinBERT (financial-domain expert) broke the tie

--- follow-up question (conversational memory) ---
USER: why did you classify that as Bullish?
For "$AMZN crushes earnings, AWS reaccelerates, shares surge after hours...":
Reasoning trace:
  1. VADER baseline -> Bearish (compound=-0.178)
  2. weak signal -> LightGBM+SBERT -> Bullish (conf=0.92)
  3. disagreement -> FinBERT tiebreaker -> Bullish (conf=0.78)
FINAL VERDICT: Bullish (class=1)
Why: VADER and LightGBM disagreed; FinBERT (financial-domain expert) broke the tie


C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
# Batch evaluation on a held-out validation sample
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

train = pd.read_csv('data/raw/train.csv')
_, X_val, _, y_val = train_test_split(
    train['text'].tolist(), train['label'].values,
    test_size=0.05, random_state=42, stratify=train['label'],
)
sample_size = min(100, len(X_val))
print(f'Evaluating the agentic workflow on {sample_size} held-out samples...')

def parse_pred(output):
    m = re.search(r'class=(\d)', output)
    return int(m.group(1)) if m else 2

agent_preds = [parse_pred(agent.invoke({'input': t})['output']) for t in X_val[:sample_size]]
f1 = f1_score(y_val[:sample_size], agent_preds, average='macro')
print(f'\nAgent (adaptive routing) F1-macro on {sample_size} samples: {f1:.4f}')
print(classification_report(y_val[:sample_size], agent_preds,
                            target_names=['Bearish', 'Bullish', 'Neutral']))

Evaluating the agentic workflow on 100 held-out samples...


C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\tiago\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Agent (adaptive routing) F1-macro on 100 samples: 0.6504
              precision    recall  f1-score   support

     Bearish       0.52      0.80      0.63        15
     Bullish       0.51      0.83      0.63        23
     Neutral       0.88      0.56      0.69        62

    accuracy                           0.66       100
   macro avg       0.64      0.73      0.65       100
weighted avg       0.74      0.66      0.67       100



## Why this is non-trivial orchestration

1. **Adaptive tool selection** - the agent chooses *which* model to consult based on VADER signal
   strength, rather than always polling all three.
2. **Disagreement detection + tie-breaking** - FinBERT (the domain expert) is invoked only when
   VADER and LightGBM disagree.
3. **Explainability** - every verdict states which model was trusted and why.
4. **Conversational memory** - follow-up questions ("why did you classify that as Bearish?") are
   answered from `ConversationBufferMemory`.

This exceeds simple ensembling/voting and is fully reproducible offline (no API key), so it can be
re-run during the oral defence. Source: `src/agent.py`.
